In [0]:
# COMMAND ----------
# 0.1 Set up dynamic workspace variables

import re

user_email = spark.sql("SELECT current_user()").collect()[0][0]

# Extract first name + first letter of last name
name_parts = re.sub(r'[^a-zA-Z0-9 ]', ' ', user_email.split('@')[0]).split()

first_name = name_parts[0].lower()
last_initial = name_parts[1][0].lower() if len(name_parts) > 1 else ""

user_prefix = f"{first_name}_{last_initial}"

# Dynamic Unity Catalog objects
CATALOG_NAME = f"data_eng_{user_prefix}"
SCHEMA_NAME = f"{user_prefix}"
VOLUME_NAME = f"raw_landing_{user_prefix}"

print(f"👤 Current User  : {user_email}")
print(f"🔑 User Prefix   : {user_prefix}")
print(f"📦 Target Catalog: {CATALOG_NAME}")
print(f"📂 Target Schema : {SCHEMA_NAME}")
print(f"🗄️ Target Volume : {VOLUME_NAME}")

In [0]:
HINTS = {
    (
        "notebook_1",
        "step_1",
    ): """
        <details class="hintbox">
          <summary>Show me the solution!</summary>
          <p>The full command is CREATE CATALOG IF NOT EXISTS `{CATALOG_NAME}`</p>
        </details>
        """,
    (
        "notebook_1",
        "step_2",
    ): """
        <details class="hintbox">
          <summary>Show me the solution!</summary>
          <p>The full command is CREATE SCHEMA IF NOT EXISTS `{SCHEMA_NAME}`</p>
        </details>
        """,
    (
        "notebook_1",
        "step_3",
        ): """
        <details class="hintbox">
          <summary>Show me the solution!</summary>
          <p>The full command is CREATE VOLUME IF NOT EXISTS `{CATALOG_NAME}`.`{SCHEMA_NAME}`.`{VOLUME_NAME}`</p>
        </details>
        """,
    (
        "notebook_1",
        "step_4.1",
        ): """
        <details class="hintbox">
          <summary>Show me a hint!</summary>
          <p>Use .dropDuplicates(["consultant_id"])</p>
        </details>
        """,
    (
        "notebook_1",
        "step_4.2",
        ): """
        <details class="hintbox">
          <summary>Show me a hint!</summary>
          <p>Use col(c).isNull()</p>
        </details>
        """,
    (
        "notebook_1",
        "step_4.3",
        ): """
        <details class="hintbox">
          <summary>Show me a hint!</summary>
          <p>Use filter() and isNull()</p>
        </details>
        """,
    (
        "notebook_1",
        "step_4.4",
        ): """
        <details class="hintbox">
          <summary>Show me a hint!</summary>
          <p>Combine col("hours") <= 0 with to_date(col("work_date"), "yyyy-MM-dd").isNull()</p>
        </details>
        """,
    (
        "notebook_1",
        "step_5.1",
        ): """
        <details class="hintbox">
          <summary>Show me a hint!</summary>
          <p>Use PySpark's built-in functions current_timestamp() and col("_metadata.file_path")</p>
        </details>
        """,
    (
        "notebook_1",
        "step_5.2",
        ): """
        <details class="hintbox">
          <summary>Show me a hint!</summary>
          <p>Use .dropDuplicates(["entry_id"]), .filter(...) to remove corrupt records, and sha2(concat_ws("||", ...), 256) to generate the SHA-256 key</p>
        </details>
        """,
     (
        "notebook_1",
        "step_5.3.1",
        ): """
        <details class="hintbox">
          <summary>Show me a hint!</summary>
          <p>Use SQL aggregate functions SUM(...) and COUNT(...). Remember, you don't need distinct since we have already deduplicated!</p>
        </details>
        """,
      (
        "notebook_1",
        "step_5.3.2",
        ): """
        <details class="hintbox">
          <summary>Show me a hint!</summary>
          <p>Use SQL AVG(...) to calculate average hours and ROUND(..., 2) to format the decimal</p>
        </details>
        """,
      (
        "notebook_2",
        "step_1.1",
        ): """
        <details class="hintbox">
          <summary>Show me the solution!</summary>
          <p>Use `{CATALOG_NAME}`.`{SCHEMA_NAME}`.`silver_projects`</p>
        </details>
        """,
      (
        "notebook_2",
        "step_1.2",
        ): """
        <details class="hintbox">
          <summary>Show me the solution!</summary>
          <p>Fill the gaps with ON t.consultant_id = c.consultant_id and t.project_code = p.project_code accordingly.</p>
        </details>
        """,
      (
        "notebook_2",
        "step_2.1",
        ): """
        <details class="hintbox">
          <summary>Show me a hint!</summary>
          <p>An active counsultant should have true is_current and null valid_to (since we don't know when they will become inactive).</p>
        </details>
        """,
      (
        "notebook_2",
        "step_2.2",
        ): """
        <details class="hintbox">
          <summary>Show me a hint!</summary>
          <p>"close" the old entry and mark as true the new one.</p>
        </details>
        """,
}

In [0]:
def load_hint(notebook, quest_id):
    base_css = """
   <style>
* {
  font-size: 16px;
  box-sizing: border-box;
}

/* Modern Expandable Container */
.hintbox {
  margin: 16px 0;
  padding: 16px 20px;
  border: 1px solid #e2e8f0;
  border-left: 4px solid #6366f1; /* Professional Indigo Accent */
  border-radius: 8px;
  background-color: #ffffff;
  box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.05), 0 2px 4px -1px rgba(0, 0, 0, 0.03);
  transition: all 0.2s ease-in-out;
}

.hintbox:hover {
  box-shadow: 0 6px 12px -2px rgba(0, 0, 0, 0.08);
}

/* Interactive Summary / Header */
.hintbox summary {
  cursor: pointer;
  font-weight: 600;
  color: #1e293b;
  user-select: none;
  outline: none;
  padding: 2px 0;
  transition: color 0.15s ease;
}

.hintbox summary:hover {
  color: #6366f1;
}

/* List Styling */
.hintbox ul {
  padding-left: 24px;
  margin-top: 12px;
  margin-bottom: 12px;
}

.hintbox ul li {
  list-style-type: disc;
  margin-bottom: 6px;
  color: #334155;
}

.hintbox ul li::marker {
  color: #6366f1; /* Soft Indigo Bullets */
}

/* Inline Code Elements */
.hintbox code {
  background: #f1f5f9;
  color: #0f172a;
  display: inline-block;
  padding: 2px 6px;
  border-radius: 4px;
  font-family: 'Fira Code', 'Consolas', monospace;
  font-size: 14px;
}

/* Highlighted Answer Class */
.answer {
  color: #4f46e5;
  font-weight: 600;
  text-decoration: none;
  border-bottom: 2px dotted #818cf8;
}

/* Code Block Styling */
.code-block {
  background: #0f172a; /* Slate Dark Mode Theme for High Contrast */
  padding: 16px 20px;
  border-radius: 8px;
  margin: 14px 0 6px 0;
  overflow-x: auto;
  border: 1px solid #1e293b;
}

.code-block pre {
  margin: 0;
  font-family: 'Fira Code', 'Cascadia Code', 'Consolas', monospace;
  font-size: 14px;
  line-height: 1.6;
  color: #e2e8f0;
}

/* Syntax Highlighting Tweaks */
.comment {
  color: #94a3b8;
  font-style: italic;
}

.keyword {
  color: #38bdf8; /* Bright Cyan */
  font-weight: 600;
}

.function {
  color: #818cf8; /* Soft Indigo */
}

.string {
  color: #34d399; /* Mint Green */
}

.variable {
  color: #f1f5f9;
}

.operator {
  color: #f43f5e;
}

.bracket {
  color: #cbd5e1;
}
</style>
    """

    body = HINTS.get((notebook, quest_id))
    if not body:
        displayHTML("No hint available.")
        return

    displayHTML(base_css + body)